# ARIMA MODEL TRAINING

**Notebook 03**: Train ARIMA models for carbon emission forecasting

## What This Does

1. Load preprocessed data
2. Select key time series for modeling
3. Train ARIMA models
4. Evaluate predictions
5. Save models and predictions

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import joblib
import json
from tqdm import tqdm

warnings.filterwarnings('ignore')
print("Libraries imported successfully")

Libraries imported successfully


## Load Preprocessed Data

In [2]:
# Load train and test data
train = pd.read_csv('../data/processed/train_data.csv')
test = pd.read_csv('../data/processed/test_data.csv')

print(f"Train data: {len(train)} records ({train['year'].min()}-{train['year'].max()})")
print(f"Test data: {len(test)} records ({test['year'].min()}-{test['year'].max()})")

Train data: 53391 records (1970-2015)
Test data: 6510 records (2016-2021)


## Select Key Series for Modeling

We'll model:
- Top 10 states (by 2021 emissions)
- Total emissions (all fuels)
- All sectors

In [3]:
# Get top 10 states
top_states = test[test['fuel-name'] == 'All Fuels'].groupby('state-name')['value'].sum().nlargest(10).index.tolist()

# Filter data for modeling
train_subset = train[
    (train['state-name'].isin(top_states)) &
    (train['fuel-name'] == 'All Fuels')
].copy()

test_subset = test[
    (test['state-name'].isin(top_states)) &
    (test['fuel-name'] == 'All Fuels')
].copy()

print(f"Selected {len(top_states)} states for modeling")
print(f"Top states: {', '.join(top_states[:5])}...")

Selected 10 states for modeling
Top states: United States, Texas, California, Florida, Pennsylvania...


## ARIMA Model Training

For each state-sector combination, we'll:
1. Fit ARIMA(1,1,1) - simple but effective
2. Make predictions on test set
3. Calculate error metrics

In [4]:
def train_arima_model(train_series, test_series, order=(1,1,1)):
    """
    Train ARIMA model and make predictions
    """
    try:
        # Fit model
        model = ARIMA(train_series, order=order)
        fitted = model.fit()
        
        # Forecast
        forecast = fitted.forecast(steps=len(test_series))
        
        # Calculate metrics
        mae = mean_absolute_error(test_series, forecast)
        rmse = np.sqrt(mean_squared_error(test_series, forecast))
        mape = mean_absolute_percentage_error(test_series, forecast) * 100
        
        return {
            'model': fitted,
            'predictions': forecast,
            'mae': mae,
            'rmse': rmse,
            'mape': mape,
            'success': True
        }
    except:
        return {'success': False}

print("ARIMA training function defined")

ARIMA training function defined


In [5]:
# Train models for each series
results = []
models_dict = {}

unique_combos = train_subset.groupby(['state-name', 'sector-name'])

print(f"Training ARIMA models for {len(unique_combos)} series...")
print("This will take 5-10 minutes...\n")

for (state, sector), group in tqdm(unique_combos, desc="Training ARIMA"):
    # Get train series
    train_series = group.sort_values('year')['value']
    
    # Get corresponding test series
    test_group = test_subset[
        (test_subset['state-name'] == state) &
        (test_subset['sector-name'] == sector)
    ].sort_values('year')
    
    if len(test_group) == 0:
        continue
    
    test_series = test_group['value']
    
    # Train model
    result = train_arima_model(train_series, test_series)
    
    if result['success']:
        # Store model
        model_key = f"{state}_{sector}_All Fuels".replace(' ', '_').replace('/', '_')
        models_dict[model_key] = result['model']
        
        # Store metrics
        results.append({
            'state': state,
            'sector': sector,
            'fuel': 'All Fuels',
            'mae': result['mae'],
            'rmse': result['rmse'],
            'mape': result['mape']
        })

results_df = pd.DataFrame(results)
print(f"\nSuccessfully trained {len(results_df)} ARIMA models")

Training ARIMA models for 60 series...
This will take 5-10 minutes...



Training ARIMA: 100%|██████████| 60/60 [00:00<00:00, 93.11it/s]


Successfully trained 60 ARIMA models


## Model Performance Summary

In [6]:
print("ARIMA Model Performance Summary")
print("=" * 50)
print(f"\nAverage MAPE: {results_df['mape'].mean():.2f}%")
print(f"Average MAE: {results_df['mae'].mean():.2f}")
print(f"Average RMSE: {results_df['rmse'].mean():.2f}")

print("\nTop 10 Best Performing Models (by MAPE):")
print(results_df.nsmallest(10, 'mape')[['state', 'sector', 'mape']])

print("\nTop 10 Worst Performing Models (by MAPE):")
print(results_df.nlargest(10, 'mape')[['state', 'sector', 'mape']])

ARIMA Model Performance Summary

Average MAPE: 7.88%
Average MAE: 14.91
Average RMSE: 18.72

Top 10 Best Performing Models (by MAPE):
            state                                           sector      mape
14       Illinois              Industrial carbon dioxide emissions  1.940460
41           Ohio          Transportation carbon dioxide emissions  2.497144
56  United States              Industrial carbon dioxide emissions  2.756820
0      California              Commercial carbon dioxide emissions  2.758008
6         Florida              Commercial carbon dioxide emissions  2.855192
52          Texas  Total carbon dioxide emissions from all sectors  2.883746
28      Louisiana  Total carbon dioxide emissions from all sectors  2.923139
38           Ohio              Industrial carbon dioxide emissions  3.153315
20        Indiana              Industrial carbon dioxide emissions  3.205424
30       New York              Commercial carbon dioxide emissions  4.104732

Top 10 Worst Perfo

## Save Models and Results

In [7]:
# Save performance metrics
results_df.to_csv('../reports/model_comparison/arima_results.csv', index=False)
print("Saved: reports/model_comparison/arima_results.csv")

# Save models (top 20 only to save space)
top_models = results_df.nsmallest(20, 'mape')

for idx, row in top_models.iterrows():
    model_key = f"{row['state']}_{row['sector']}_All Fuels".replace(' ', '_').replace('/', '_')
    if model_key in models_dict:
        filepath = f"../models/arima/{model_key}.pkl"
        joblib.dump(models_dict[model_key], filepath)

print(f"Saved top 20 ARIMA models to models/arima/")

# Save summary
summary = {
    'model_type': 'ARIMA',
    'n_models_trained': len(results_df),
    'average_mape': float(results_df['mape'].mean()),
    'average_mae': float(results_df['mae'].mean()),
    'average_rmse': float(results_df['rmse'].mean()),
    'best_mape': float(results_df['mape'].min()),
    'worst_mape': float(results_df['mape'].max())
}

with open('../models/arima/arima_summary.json', 'w') as f:
    json.dump(summary, f, indent=4)
print("Saved: models/arima/arima_summary.json")

Saved: reports/model_comparison/arima_results.csv
Saved top 20 ARIMA models to models/arima/
Saved: models/arima/arima_summary.json


## Generate Future Predictions (2022-2030)

For dashboard visualization

In [8]:
# Generate predictions for 2022-2030
future_predictions = []

for (state, sector), group in tqdm(train_subset.groupby(['state-name', 'sector-name']), 
                                    desc="Generating forecasts"):
    # Get full series (train + test)
    full_series = pd.concat([
        train_subset[(train_subset['state-name'] == state) & (train_subset['sector-name'] == sector)],
        test_subset[(test_subset['state-name'] == state) & (test_subset['sector-name'] == sector)]
    ]).sort_values('year')['value']
    
    try:
        # Fit on all available data
        model = ARIMA(full_series, order=(1,1,1))
        fitted = model.fit()
        
        # Forecast 2022-2030 (9 years)
        forecast = fitted.forecast(steps=9)
        
        for i, year in enumerate(range(2022, 2031)):
            future_predictions.append({
                'year': year,
                'state': state,
                'sector': sector,
                'fuel': 'All Fuels',
                'predicted_value': forecast.iloc[i],
                'model': 'ARIMA'
            })
    except:
        pass

predictions_df = pd.DataFrame(future_predictions)
predictions_df.to_csv('../data/dashboard/arima_predictions_2022_2030.csv', index=False)
print(f"\nSaved {len(predictions_df)} future predictions for dashboard")
print("File: data/dashboard/arima_predictions_2022_2030.csv")

Generating forecasts: 100%|██████████| 60/60 [00:00<00:00, 89.31it/s]


Saved 540 future predictions for dashboard
File: data/dashboard/arima_predictions_2022_2030.csv


## Notebook Complete

In [9]:
print("=" * 70)
print("ARIMA MODEL TRAINING COMPLETE")
print("=" * 70)
print(f"\nModels trained: {len(results_df)}")
print(f"Average MAPE: {results_df['mape'].mean():.2f}%")
print(f"\nFiles saved:")
print("  - reports/model_comparison/arima_results.csv")
print("  - models/arima/*.pkl (top 20 models)")
print("  - models/arima/arima_summary.json")
print("  - data/dashboard/arima_predictions_2022_2030.csv")
print("\nNext: Run 04_prophet_model.ipynb")

ARIMA MODEL TRAINING COMPLETE

Models trained: 60
Average MAPE: 7.88%

Files saved:
  - reports/model_comparison/arima_results.csv
  - models/arima/*.pkl (top 20 models)
  - models/arima/arima_summary.json
  - data/dashboard/arima_predictions_2022_2030.csv

Next: Run 04_prophet_model.ipynb
